In [ ]:
from langgraph.graph import StateGraph ,START,END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from  dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage,BaseMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
import operator
import time

In [ ]:
class Crashstate(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str

In [ ]:
#dwfine steps
def step_1(state: Crashstate) -> Crashstate:
    print("✅ Step1 executed")
    return {"step1":"done","input":state['input']}

def step_2(state: Crashstate) -> Crashstate:
    print(" step2 hanging ... now manually from the notebook toolbar (Stop button)")
    time.sleep(30)
    return {"step2": "done"}

def step_3(state: Crashstate) -> Crashstate:
    print("✅ Step3 executed")
    return {"step3":"done"}

In [ ]:
graph = StateGraph(Crashstate)
graph.add_node("step_1",step_1)
graph.add_node("step_2",step_2)
graph.add_node("step_3",step_3)
graph.add_edge(START,"step_1")
graph.add_edge("step_1","step_2")
graph.add_edge("step_2","step_3")
graph.add_edge("step_3",END)

checkpointer = InMemorySaver()

workflow=graph.compile(checkpointer=checkpointer)



In [ ]:
try:
    print("Running graph: please manually interrupt during step2..")
    workflow.invoke({"input": "start"},config={"configurable": {"thread_id": "thread-1"}})
except KeyboardInterrupt:
    print("❌ kernel manually interrupted (crash simulated).")

Running graph: please manually interrupt during step2..
✅ Step1 executed
 step2 hanging ... now manually from the notebook toolbar (Stop button)
❌ kernel manually interrupted (crash simulated).


In [ ]:
workflow.get_state({"configurable":{"thread_id":"thread-1"}})

StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18b0c6-559e-654a-8001-1dec263eece1'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-07-29T05:14:38.145449+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18b0c6-559a-6e0d-8000-e85c21807104'}}, tasks=(PregelTask(id='3d8be79d-4e2a-1cee-0476-56c0ceaf94cb', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [ ]:
list(workflow.get_state_history({"configurable":{"thread_id":"thread-1"}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18b0c6-559e-654a-8001-1dec263eece1'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-07-29T05:14:38.145449+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18b0c6-559a-6e0d-8000-e85c21807104'}}, tasks=(PregelTask(id='3d8be79d-4e2a-1cee-0476-56c0ceaf94cb', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start'}, next=('step_1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18b0c6-559a-6e0d-8000-e85c21807104'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-07-29T05:14:38.144036+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'ch

In [ ]:
final_state = workflow.invoke(None,config={"configurable":{"thread_id":"thread-1"}})
print("\n✅ Final State:",final_state)

 step2 hanging ... now manually from the notebook toolbar (Stop button)
✅ Step3 executed

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}


In [ ]:
workflow.get_state({"configurable":{"thread_id":"thread-1"}})

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18b0da-4278-639c-8003-1154053a304c'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-29T05:23:33.008453+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18b0da-4274-6013-8002-e34a5c31bddb'}}, tasks=(), interrupts=())

In [ ]:
list(workflow.get_state_history({"configurable":{"thread_id":"thread-1"}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18b0da-4278-639c-8003-1154053a304c'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-29T05:23:33.008453+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18b0da-4274-6013-8002-e34a5c31bddb'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18b0da-4274-6013-8002-e34a5c31bddb'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-29T05:23:33.006706+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18b0c6-559e-654a-8001-1dec263eece1'}}, tasks=(PregelTask(id='f75db3e9-34dd-137a